# Invoice Flagging

SQL, exploratory analysis and classification of invoice records using rule-based risk flags.

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
conn=sqlite3.connect(r"../inventory.db")
tables=pd.read_sql_query("select name from sqlite_master where type='table'",conn)

In [ ]:
for table in tables['name']:
    print("Table Name",table)
    df=pd.read_sql_query(f"select * from {table} limit 5",conn)
    display(df)

## Purchase and invoice aggregation

In [ ]:
purchases_agg=pd.read_sql_query("""select PONumber,count(distinct Brand) as total_brand
,sum(Quantity) as total_quantity,sum(Dollars) as total_dollar,avg(julianday(ReceivingDate)-julianday(PODate)) as avg_receving_dely from purchases group by PONumber""",conn)

In [ ]:
pd.read_sql_query("""select PONumber, Quantity as invoice_quantity,Dollars as invoice_dollars,Freight,julianday(InvoiceDate)-julianday(PODate) as days_po_to_invoice,julianday(PayDate)-julianday(InvoiceDate) as days_to_pay from vendor_invoice""",conn)

In [ ]:
df=pd.read_sql_query("""with purchase_agg as (select PONumber,count(distinct Brand) as total_brand,sum(Quantity) as total_quantity,sum(Dollars) as total_dollar,avg(julianday(ReceivingDate)-julianday(PODate)) as avg_receving_dely from purchases group by PONumber) select v.PONumber,Quantity as invoice_quantity,Dollars as invoice_dollars,Freight,julianday(InvoiceDate)-julianday(PODate) as days_po_to_invoice,julianday(PayDate)-julianday(InvoiceDate) as days_to_pay,total_dollar,total_brand,total_quantity,avg_receving_dely from vendor_invoice v left join purchase_agg p on p.PONumber=v.PONumber""",conn)

In [ ]:
df.isnull().sum()
df

## Rule-based invoice flag

In [ ]:
def invoice_risk(row):
    if abs(row['invoice_dollars']-row['total_dollar'])>5:
        return 1
    if row['avg_receving_dely']>10:
        return 1
    return 0

df['flagged_invoice']=df.apply(invoice_risk,axis=1)
df['flagged_invoice'].value_counts()

In [ ]:
df['flagged_invoice'].value_counts().plot(kind='bar')
plt.title('Invoice Flag Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(df.corr(numeric_only=True),annot=True)
plt.show()

## Statistical comparison

In [ ]:
from scipy.stats import ttest_ind

flagged=df[df['flagged_invoice']==1]
normal=df[df['flagged_invoice']==0]
significant_features=[]
non_significant_features=[]
results=[]
metrics=['invoice_quantity','invoice_dollars','Freight','days_po_to_invoice','days_to_pay','total_brand','total_quantity','total_dollar','avg_receving_dely']

for metric in metrics:
    flagged_mean=flagged[metric].mean()
    normal_mean=normal[metric].mean()
    t_stat,p_value=ttest_ind(flagged[metric].dropna(),normal[metric].dropna(),equal_var=False)
    if p_value<0.05:
        significant_features.append(metric)
        results.append({'metrics':metric,'flagged_mean':round(flagged_mean,2),'normal_mean':round(normal_mean,2),'p_value':round(p_value,3)})
    else:
        non_significant_features.append(metric)

In [ ]:
print('Significant features:', significant_features)
print('Non-significant features:', non_significant_features)
results

## Classification experiments

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X=df[['invoice_quantity','avg_receving_dely','Freight','total_quantity','total_dollar']]
y=df['flagged_invoice']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
scaler=StandardScaler()
x_trained=scaler.fit_transform(X_train)
x_tested=scaler.transform(X_test)

In [ ]:
def detect(model,x_test,y_test,model_name):
    pred=model.predict(x_test)
    print(f'\nModel: {model_name}')
    print(classification_report(y_test,pred))

model1=LogisticRegression(random_state=42)
model1.fit(x_trained,y_train)
detect(model1,x_tested,y_test,'Logistic Regression')

model2=DecisionTreeClassifier(random_state=42)
model2.fit(x_trained,y_train)
detect(model2,x_tested,y_test,'Decision Tree')

model3=RandomForestClassifier(random_state=42)
model3.fit(x_trained,y_train)
detect(model3,x_tested,y_test,'Random Forest')

In [ ]:
feature_importance=pd.DataFrame({'Feature':X_train.columns,'importance':model3.feature_importances_}).sort_values(by='importance',ascending=False)
feature_importance

## Random Forest tuning

In [ ]:
from sklearn.metrics import make_scorer,f1_score
from sklearn.model_selection import GridSearchCV

rf=RandomForestClassifier(random_state=42,n_jobs=-1)
param_grid={'n_estimators':[100,200,300],'max_depth':[None,4,5,6],'min_samples_split':[2,3,5],'min_samples_leaf':[1,2,5],'criterion':['gini','entropy']}
scorer=make_scorer(f1_score)
grid_search=GridSearchCV(estimator=rf,param_grid=param_grid,scoring=scorer,cv=5,verbose=2,n_jobs=-1)
grid_search.fit(x_trained,y_train)
print('Best Parameters:')
print(grid_search.best_params_)
print('\nBest F1 Score:')
print(grid_search.best_score_)
best_model=grid_search.best_estimator_
y_pred=best_model.predict(x_tested)
test_f1=f1_score(y_test,y_pred)
print('\nTest F1 Score:')
print(test_f1)